# LLM Metrics: как измеряют языковые модели

Эта тема стоит несколько особняком в обзоре. Все остальные темы рассказывают, как устроены модели и как они учатся. Эта тема — про то, как мы понимаем, что одна модель лучше другой. На первый взгляд вопрос кажется техническим и второстепенным, но на практике именно метрики задают направление всему развитию области: модели оптимизируют то, что мы умеем измерять. История метрик — это история того, как менялось само представление о том, что значит «хорошая языковая модель»: от «насколько текст правдоподобен» до «насколько ответ верен, честен и полезен» и, наконец, «способна ли модель действовать».

Удобно держать в голове общую линию эволюции, к которой мы будем возвращаться:

внутренние метрики (перплексия) → метрики совпадения с эталоном (BLEU, ROUGE) → семантические и модельные метрики (BERTScore, LLM-судья) → бенчмарки отдельных задач → агрегированные наборы (GLUE, MMLU) → человеческие предпочтения и арены → устойчивые к загрязнению, агентные и длинноконтекстные оценки.

## Почему измерять языковые модели трудно

Главная сложность в том, что язык открыт. У арифметической задачи есть один правильный ответ, а у запроса «перескажи этот текст» или «напиши письмо коллеге» правильных ответов бесконечно много, и они могут не совпадать ни одним словом. Поэтому привычная для классификации логика «угадал / не угадал» к генерации напрямую не применяется, и значительная часть истории метрик — это попытки обойти эту проблему.

Полезно сразу разделить две разные задачи, которые часто путают. Первая — отслеживать прогресс во время обучения: здесь нужна дешёвая, автоматическая, чувствительная метрика, которую можно считать на каждом шаге. Вторая — сравнивать готовые модели по их способностям: здесь важна не дешевизна, а то, насколько метрика отражает реальную полезность. Внутренние метрики хорошо решают первую задачу, бенчмарки и человеческая оценка — вторую.

И ещё один сквозной мотив, к которому мы вернёмся в конце: как только метрика становится целью, она перестаёт быть хорошей метрикой. Это закон Гудхарта, и в области LLM он работает особенно жёстко.

## Внутренние метрики: перплексия и кросс-энтропия

Языковая модель в своей основе — это модель вероятности следующего токена. Она присваивает каждой последовательности вероятность, разложенную по цепочке условных вероятностей каждого токена при условии предыдущих. Естественная мера качества такой модели — насколько высокую вероятность она присваивает реальному тексту, которого раньше не видела. Если модель хорошо понимает язык, настоящие тексты должны быть для неё «ожидаемыми».

Формально это выражается через кросс-энтропию — среднее отрицательное лог-правдоподобие токенов на отложенной выборке:

```
H = -(1/N) Σ log p(x_i | x_1, ..., x_{i-1})
```

Перплексия — это просто экспонента от кросс-энтропии:

```
PPL = exp(H)
```

Интуиция за перплексией следующая: это «эффективное число равновероятных вариантов», между которыми модель в среднем колеблется на каждом шаге. Перплексия 1 означала бы идеальное предсказание (модель всегда уверена в правильном токене), а перплексия, равная размеру словаря, — полное незнание языка (модель угадывает наугад). Чем ниже перплексия, тем лучше. Историческими ориентирами служили, например, наборы вроде Penn Treebank и WikiText, на которых десятилетиями мерили прогресс языкового моделирования.

У перплексии есть важная техническая оговорка. Она зависит от токенизации и словаря, поэтому перплексии двух моделей с разными токенизаторами напрямую несравнимы. Чтобы обойти это, используют метрики, нормированные на символы или байты — bits-per-character и bits-per-byte, — которые не зависят от того, как именно текст разбит на токены, и позволяют честно сравнивать разные архитектуры.

Сильные стороны перплексии — дешевизна и отсутствие необходимости в разметке: её можно считать на любом корпусе и строить по ней кривые обучения. Именно поэтому она остаётся главной метрикой на этапе предобучения. Слабость в том, что перплексия измеряет правдоподобие и беглость, но не полезность. Модель может прекрасно предсказывать токены и при этом плохо следовать инструкциям, врать или быть бесполезной в диалоге. Поэтому с переходом к инструктивным и диалоговым моделям перплексия перестала быть достаточной, и центр тяжести сместился к внешним, поведенческим метрикам.

## Метрики качества генерации относительно эталона

Первый исторический ответ на проблему открытости генерации возник в машинном переводе и реферировании. Идея простая: дать модели вход, получить выход и сравнить его с одним или несколькими эталонными ответами, написанными человеком. Качество сводится к мере совпадения с эталоном.

BLEU, предложенная для оценки перевода, измеряет точность совпадения n-грамм между выходом модели и эталоном, добавляя штраф за слишком короткие ответы. ROUGE, возникшая для реферирования, наоборот, ориентирована на полноту: насколько n-граммы и общие подпоследовательности эталона покрыты в ответе модели. Рядом стоит целое семейство похожих метрик: METEOR, учитывающая синонимы и словоформы, и chrF, работающая на символьных n-граммах. В задачах с коротким фактическим ответом (например, вопросно-ответные системы на SQuAD) используют точное совпадение и токенную F-меру.

Эти метрики дёшевы, воспроизводимы и автоматизируемы, поэтому они десятилетиями были стандартом. Но у них есть фундаментальный изъян: они меряют поверхностное совпадение слов, а не смысл. Корректный перефраз, не совпадающий с эталоном лексически, получает низкую оценку, а бессмысленный текст с правильными словами — завышенную. Корреляция таких метрик с человеческим суждением для по-настоящему открытой генерации слабая. Осознание этого изъяна и подтолкнуло переход к следующему семейству.

## Метрики на основе эмбеддингов и моделей

Логичный следующий шаг — сравнивать не строки, а смыслы. Для этого метрики стали опираться на представления, выученные самими нейросетями.

BERTScore сопоставляет токены ответа и эталона по близости их контекстных эмбеддингов, благодаря чему улавливает синонимию и перефразирование, недоступные подсчёту n-грамм. Ещё дальше идут обучаемые метрики — BLEURT и COMET (последняя стала фактическим стандартом в машинном переводе): это отдельные модели, обученные на человеческих оценках качества, то есть метрика буквально предсказывает, как ответ оценил бы человек. Здесь происходит концептуальный сдвиг: метрика сама становится моделью.

Кульминация этой линии — подход «LLM как судья» (LLM-as-a-judge), который к середине 2020-х стал доминирующим способом оценки открытой генерации. Сильная языковая модель получает запрос, ответ (или пару ответов) и инструкцию выставить оценку или выбрать лучший вариант. Это масштабируется несравнимо лучше человеческой разметки и заметно лучше n-граммных метрик коррелирует с мнением людей. Но у судьи-модели есть свои систематические искажения, о которых важно знать: предпочтение собственного стиля и собственных ответов, смещение в сторону более длинных и уверенно звучащих ответов, чувствительность к порядку предъявления вариантов. Поэтому LLM-судью обычно калибруют, перемешивают порядок и перепроверяют на части примеров людьми.

## Метрики на уровне задач

Параллельно с метриками генерации всегда существовали и простые метрики для задач с проверяемым ответом, и именно они лежат в основе большинства современных бенчмарков. Для классификации это точность (accuracy), а также precision, recall и F-мера. Для кода ключевой метрикой стала pass@k: сгенерированный код запускают на модульных тестах и проверяют функциональную корректность, а pass@k оценивает вероятность того, что среди k попыток хотя бы одна пройдёт все тесты. Для математики используют точное совпадение нормализованного финального ответа. Для сравнительных оценок — долю побед (win rate) одной модели над другой.

Объединяет это семейство то, что у задачи есть объективно проверяемый ответ, и оценка не требует ни эталонного текста, ни судьи. Это делает такие метрики надёжными — и именно поэтому современный фронтир всё больше смещается в сторону задач (код, математика, агентные сценарии), где корректность можно проверить автоматически и однозначно.

## Бенчмарки: от отдельных задач к наборам

Бенчмарк — это стандартизованный набор данных и протокол оценки, позволяющий сравнивать модели на равных. История бенчмарков хорошо показывает, как росли амбиции области.

Сначала была эпоха отдельных задач: каждый датасет (SQuAD для вопросов-ответов, SNLI для логического следования и так далее) мерил одну узкую способность. Затем пришла эпоха агрегации: бенчмарки GLUE и его усложнённый наследник SuperGLUE собрали россыпь задач на понимание языка в единый набор с одной сводной цифрой, чтобы измерять «понимание языка вообще». Характерно, что обе оценки были вскоре «решены» — модели превысили человеческий уровень, и это стало повторяющимся сюжетом.

Следующая эпоха — знания и рассуждения. Её определяющим бенчмарком стал MMLU: 57 предметов от школьного до профессионального уровня, от анатомии до юриспруденции. Рядом встали бенчмарки здравого смысла (HellaSwag, ARC, WinoGrande, PIQA) и правдивости (TruthfulQA, проверяющий, повторяет ли модель распространённые человеческие заблуждения). Отдельно развивались математика (GSM8K — школьные задачи, MATH — олимпиадные) и код (HumanEval, MBPP).

Параллельно возникли мега-наборы и идея холистической оценки. BIG-bench собрал более двухсот разнообразных задач, придуманных сообществом; из него выделили особо трудное подмножество BIG-bench Hard. Проект HELM сместил акцент с одной цифры на многомерность: модель прогоняют по множеству сценариев и меряют не только точность, но и устойчивость, калибровку, смещения, эффективность. Это была важная смена философии — от «кто набрал больше» к «какова модель по совокупности свойств».

К середине 2020-х область столкнулась с кризисом насыщения. Классические бенчмарки — MMLU, HellaSwag, HumanEval — фронтирные модели стали проходить выше 90%, и различия между ними утонули в шуме. Насыщенный бенчмарк перестаёт что-либо различать. Ответом стало новое поколение более трудных оценок:

- MMLU-Pro — усложнённый MMLU с десятью вариантами ответа вместо четырёх и обязательной цепочкой рассуждений (хотя к 2026 году и он подходит к насыщению).
- GPQA-Diamond — вопросы уровня PhD по биологии, физике и химии, специально составленные так, чтобы их нельзя было нагуглить; неспециалисты набирают около трети даже с доступом в интернет.
- Humanity's Last Exam — около трёх тысяч вопросов экспертного уровня от специалистов разных областей, задуманные так, чтобы оставаться трудными несколько лет.
- ARC-AGI и ARC-AGI-2 — задачи на абстракцию и обобщение, нацеленные на «текучий интеллект», а не на эрудицию; первая версия была фактически взята reasoning-моделями к концу 2024 года, что и потребовало второй.
- Олимпиадная математика (AIME и подобные) и проекты вроде FrontierMath для самого верхнего уровня.

Отдельной и быстро растущей ветвью стали агентные бенчмарки, проверяющие не текст, а действие: SWE-bench и SWE-bench Verified (модель должна решить реальную задачу из репозитория на GitHub так, чтобы прошли тесты), а также GAIA, WebArena, AgentBench и tau-bench, оценивающие работу с инструментами, навигацию и многошаговые сценарии. Это закономерный итог общей линии: по мере того как модели становятся агентами, оценка тоже переходит от «что модель говорит» к «что модель делает».

## Типы бенчмаркинга

Полезно держать в голове, что бенчмарки различаются сразу по нескольким независимым осям. Понимание этих осей помогает и при чтении новых статей, и на собеседовании, где часто просят систематизировать, а не перечислить.

По измеряемой способности. Знания и эрудиция; рассуждения; здравый смысл; математика; код; правдивость и безопасность; многоязычность; мультимодальность; работа с инструментами и агентность; длинный контекст. Это самое привычное деление, и именно вдоль него растут новые бенчмарки.

По протоколу предъявления. Здесь важен исторический сдвиг. В эпоху GLUE модель дообучали под каждую задачу и мерили дообученную версию. С появлением больших моделей перешли к оценке без дообучения, через формулировку запроса: zero-shot (без примеров), few-shot (несколько примеров прямо в контексте) и с явной просьбой рассуждать пошагово (chain-of-thought). Один и тот же бенчмарк может давать очень разные числа в зависимости от протокола, поэтому сравнивать модели корректно только в одинаковых условиях.

По способу выставления оценки. Автоматическая проверка по совпадению или тестам (дёшево и воспроизводимо, но применимо не везде); оценка моделью-судьёй (масштабируемо, но со своими искажениями); человеческая оценка (наиболее достоверна, но дорога и медленна).

По статичности. Классический бенчмарк — это фиксированный тестовый набор. Но фиксированный набор рано или поздно утекает в обучающие данные и теряет ценность. Поэтому возникли живые и состязательные форматы: непрерывно обновляемые лидерборды и подходы вроде Dynabench, где люди специально придумывают примеры, на которых текущие модели ошибаются.

По размерности результата. Одна сводная цифра (удобно для ранжирования, но скрывает компромиссы) против холистической многомерной оценки в духе HELM (точность, устойчивость, смещения, эффективность по отдельности).

Отдельно стоит выделить оценку по человеческим предпочтениям в формате арены, потому что это качественно иной подход. Вместо фиксированных задач с известными ответами он измеряет, какой ответ людям субъективно нравится больше. Самый известный пример — LMArena (ранее известная как LMSYS Chatbot Arena): пользователю показывают ответы двух анонимных моделей на его собственный запрос, он выбирает лучший, а из миллионов таких попарных сравнений строится рейтинг по схеме Эло или модели Брэдли–Терри. Сила арены в том, что она улавливает «ощущение полезности», которое не видят формальные бенчмарки. Слабость — в том же: люди склонны голосовать за более длинные, уверенные и красиво оформленные ответы, поэтому стиль может побеждать точность, и арена меряет предпочтение, а не истину.

## Needle in a Haystack

Этот тест заслуживает отдельного разбора, потому что он возник как ответ на конкретный технологический сдвиг — взрывной рост окна контекста. Когда модели стали принимать 128 тысяч, 200 тысяч, а затем и миллионы токенов, встал естественный вопрос: модель действительно использует весь этот контекст или только его края? Заявленный размер окна и реальная способность находить в нём информацию — разные вещи.

Идея теста (его предложил Грег Камрадт в 2023 году) предельно наглядна. В длинный нейтральный текст («стог сена» — обычно эссе Пола Грэма) вставляют одно постороннее предложение («иголку»), например, утверждение, что лучшее занятие в Сан-Франциско — съесть сэндвич в парке Долорес в солнечный день. Затем модель просят ответить на вопрос, ответ на который содержится только в этой иголке. Эксперимент повторяют, систематически меняя две вещи: глубину, на которую спрятана иголка (от начала к концу документа, от 0% до 100%), и общую длину контекста. Результат рисуют тепловой картой: по одной оси — длина, по другой — глубина, цвет ячейки показывает, нашла ли модель иголку.

Такие карты обнажили несколько характерных явлений. Главное из них — эффект «потерянного в середине» (lost in the middle): информацию в начале и в конце контекста модели находят почти безошибочно, а вот ближе к середине провал в качестве. Кроме того, способность к извлечению в целом деградирует с ростом длины. Показателен ранний результат с Claude 2.1, который сначала набрал низкую точность; впоследствии выяснилось, что дело отчасти в формулировке: модель, обученная не отвечать на основании информации, которую считает необоснованной, осторожничала, и переформулировка запроса заметно меняла результат. Это хороший урок о том, что результаты бенчмарков чувствительны к деталям постановки.

При всей наглядности у одно-иголочного теста есть серьёзные ограничения. По сути это проверка извлечения одного факта, а не рассуждения, и современные сильные модели проходят его почти идеально, так что различать их он перестал. Он ничего не говорит о способности связывать несколько разнесённых фактов или рассуждать поверх длинного контекста. Поэтому тест эволюционировал в сторону усложнения. Появились варианты с несколькими иголками (multi-needle), требующие интегрировать разрозненную информацию. Возник бенчмарк RULER с набором синтетических длинноконтекстных задач — и его ключевой вывод очень показателен: модели, идеально проходящие классический Needle in a Haystack, заметно проседают на более сложных задачах RULER по мере роста длины, не справляясь с отвлекающими фрагментами и сбиваясь на копирование из контекста или на собственные знания вместо рассуждения. В том же направлении работают NeedleBench, BABILong (рассуждение в длинном контексте), NoLiMa (поиск без буквального совпадения слов) и LongBench. Общая мораль: пройденный одно-иголочный тест — необходимое, но далеко не достаточное условие настоящей работы с длинным контекстом.

## Проблемы и ограничения метрик

Любой разговор о метриках стоит заканчивать перечнем их слабых мест, потому что наивная вера в цифры лидербордов — частая ошибка.

Загрязнение данных (data contamination) — главная беда. Тестовые наборы публичны и со временем попадают в обучающие данные следующих моделей. Тогда высокий балл отражает не способность рассуждать, а запоминание ответов. Именно поэтому так ценятся свежие, приватные и состязательные оценки.

Насыщение. У каждого бенчмарка ограниченный срок жизни: как только модели упираются в его потолок, он перестаёт различать сильнейших, и нужен новый, более трудный. Мы видели это на всей цепочке от GLUE до MMLU-Pro.

Закон Гудхарта. Когда бенчмарк становится целью оптимизации, под него начинают подгонять обучение, и высокий балл может расходиться с реальной полезностью. Модель учат «сдавать экзамен», а не быть умной.

Валидность конструкта. Не всегда очевидно, что бенчмарк измеряет именно ту способность, которую заявляет; формат с выбором из вариантов, например, оставляет лазейки, которые модель может эксплуатировать, не понимая сути.

Разрыв с реальностью. Высокий балл на академическом бенчмарке не гарантирует пользы в реальном продукте; корреляция между лидербордами и фактическим качеством работы бывает слабой, и нередко модели с более скромными общими баллами оказываются точнее на конкретных прикладных задачах.

Воспроизводимость. Результаты чувствительны к формулировке запроса, числу примеров, версии оценочного инструментария и способу нормализации ответа, поэтому числа из разных источников не всегда сравнимы напрямую.

## Как это всё связано: эволюция мысли

Если свести тему к одной нити, она такая. Сначала качество модели означало правдоподобие текста, и мерили его перплексией. Затем, с приходом задач генерации, спросили о совпадении с эталоном (BLEU, ROUGE), а осознав, что слова не равны смыслу, перешли к семантическим и обучаемым метрикам и, наконец, к оценке самой моделью-судьёй. Параллельно росли бенчмарки: от отдельных задач к агрегированным наборам, от узких навыков к знаниям и рассуждению, от одной цифры к холистической оценке. Столкнувшись с насыщением и загрязнением, область двинулась к более трудным, приватным и состязательным оценкам, к человеческим аренам и к проверке не слов, а действий и работы с длинным контекстом.

За всем этим стоит один мета-сюжет: по мере того как модели становились сильнее, вопрос менялся с «насколько текст вероятен и беглый» на «насколько ответ верен, честен, полезен — и способна ли модель действовать», а сам оценщик всё чаще становился либо моделью, либо толпой людей. Понимание этой траектории важнее, чем знание конкретных аббревиатур: бенчмарки устаревают каждый год, а логика, по которой одни оценки сменяют другие, остаётся.